## Quicksort versione Las Vegas e Monte Carlo - Implementazione e Analisi
L'Algoritmo $\text{Quicksort(S)}$ affronta il classico problema dell'**ordinamento di una sequenza** $S$ di $n$ **numeri**. Le sue **prestazioni** dipendono dall'**ordinamento iniziale** di $S$; nel **caso peggiore**, nel quale si verificano partizionamenti della sequenza sfavorevoli, l'algoritmo **deterministico** ha una complessita' temporale $O(n^2)$. Di seguito vengono proposte due versioni **randomizzate** dell'algoritmo che, con piu' o meno successo, tentano di allontanarsi dal **caso peggiore** e ottengono un numero atteso di confronti pari a $O(nlogn)$.

Il **determinismo** del primo algoritmo si trova nella _scelta esplicita_ di quale elemento considerare come _pivot_, ossia, ad ogni iterazione dell'algoritmo, quale elemento $p$ usiamo come base delle _partizioni_ $S_<, p, S_{\geq}$. Un esercizio che abbiamo visto a lezione e' stato quello di creare la **sequenza peggiore** per una determinata posizione del _pivot_, infatti proprio per la _scelta esplicita_ di quest'ultima e' facile capire come generarla. Ad esempio per la versione **deterministica** di $\text{Quicksort(S)}$ che sceglie come _pivot_ l'elemento _mediano_ e' sufficente scrivere gli $n$ numeri inserendo nella posizione _mediana_ il numero piu' _piccolo_ (o piu' grande) della sequenza ancora da ordinare.


Ora passiamo al codice, inizio impostando l'ambiente e definendo la sequenza $S$ da analizzare, con $|S| = 10^4$

In [ ]:
# Imports
import IPython
import warnings

import random
import math

PATH = "../data/"
N = 1000

In [ ]:
# returns a list of n random int between 0 and 100
def sequence_generator(n):
    s = []
    for i in range(n):
        s.append(random.randint(0, 100))
    return s
# helper swap function
def swap(s, a, b):
    tmp = s[b]
    s[b] = s[a]
    s[a] = tmp
    
s = sequence_generator(N) # random sequence of int

### Las Vegas Quicksort:
Procediamo ora a definire $\text{LVQuicksort(S)}$, prima versione **randomizzata** derivata dall'omonimo **deterministico** che, al posto di effettuare una _scelta irrevocabile esplicita_ del _pivot_, ad ogni chiamata lo campiona con _probabilita' uniforme_ su tutta la sequenza ancora da ordinare. \
L'algoritmo presentato e' considerato di tipo **Las Vegas** in quanto _fornisce sempre l'output corretto anche se, sullo stesso input e' eseguito in istanti diversi_ $t_1, t_2| t_1 \neq t_2$. Ad ogni chiamata l'algoritmo si comportera' esattamente coem la versione deterministica e percio' nel **caso peggiore** costera' sempre $O(n^2)$. Per stimare il numero atteso di confronti $X$ nel caso migliore invece, considerate delle variabili indicatrici $I(i,j)$ che indicheranno se $i,j$ vengono confrontati, equivale a calcolare il valore atteso $\mathbb{E}[X] = \mathbb{E}[\sum_{i=1}^{n-1} \sum_{j=i+1}^{n} I(i,j)]$ che, per la _proprieta' della linearita' del valore atteso_, se $p_{ij}(1)$ e' la probabilita' che $i$ e $j$ _siano confrontati_ il **valore atteso del numero di confronti** $X$ diventa $\mathbb{E}[X] = =\sum_{i=1}^{n-1} \sum_{j=i+1}^{n} p_{ij}(1)$. Si hanno soli due casi favorevoli di confrontare $i$ e $j$, $I(i,j)$ e $I(j,i)$, dunque $p_{ij} = \frac{2}{j-1+1}$. Seguendo le note, per $n$, numero di elementi della sequenza, **grande** il **valore atteso del numero di confronti** e' approssimabile a $\mathbb{E}[X] = O(nlogn)$.

Ricapitolando, $\text{LVQuicksort(S)}$, come nel caso **deterministico**, avra' complessita $T_\text{best} = O(nlogn)$ e $T_\text{worst} = O(n^2)$. Il vantaggio di quest'algoritmo deriva dalla scelta **randomica** del _pivot_ e dell'impossibilita' di creare sequenze pessime prevedibili. \
Propongo un'implementazione dell'algoritmo $\text{LVQuicksort(s)}$ basato su quello classico ${\footnotesize{\text{(Recuperato da un laboratorio di ASD)}}}$.

In [ ]:
# Las Vegas Quicksort
count = 0
limit = float('inf')

def r_quick_sort(s, start, end):
    global count, limit
    if start < end:
        pivot = random.randint(start, end) # random index
        swap(s, start, pivot) # the pivot is now a the start of the partition
        # partition
        i = start+1 
        j=i
        while j <= end:
            count += 1
            if count >= limit: return False
            if s[j] < s[pivot]:
                swap(s, i, j)
                i+=1
            j+=1
        swap(s, pivot, i-1)
        pivot = i-1
        # recursive calls
        if pivot > start: 
            if not r_quick_sort(s, start, pivot-1): return False
        if pivot < end: 
            if not r_quick_sort(s, pivot+1, end): return False
    return True
    
def lv_quicksort(s):
    global count, limit
    
    count = 0
    limit = float('inf')
    if len(s) <= 1:
        return count # number of comparisons
    else:
        r_quick_sort(s, 0, len(s) - 1)
        return count

L'implementazione presentata ritorna il **numero totale di confronti effettuati** per ordinare la sequenza $s$ grazie ad un banale contatore. \
L'algoritmo $\text{LVQuicksort(S)}$, come sopra scritto, ha la stessa complessita' della versione **deterministica**, nel caso peggiore avra' quindi un costo di $O(n^2)$, vediamo adesso un modo, sfruttando la _teoria della probabilita'_, di minimizzare la frequenza del caso $T_\text{worst}$.

### Monte Carlo Quicksort:
Abbiamo detto che scelte casuali del pivot possono, con bassa probabilita', richiedere $O(n^2)$ confronti. \
Definiamo quindi $\text{MCQuicksort(S, k)}$, algoritmo di tipo **Monte Carlo** che non fa altro che eseguire $k$ volte $\text{LVQuicksort(S)}$ (con qualche accortezza) per ottenere una complessita', ossia il **numero di confronti**, nel caso peggiore $T_\text{worst} = O(nlogn)$.
Essendo pero' un algoritmo di tipo **Monte Carlo**, $\text{MCQuicksort(S, k)}$ potrebbe fallire e non ordinare la sequenza $S$ (il perche' sara' chiarito in seguito). Cio' puo' avvenire con una probabilita' che, scelto un $k$ opportuno, puo' esser resa _infinitesimamente_ piccola.

Ricordando la _disuguaglianza di Markow_: 

$P(X\geq a)\leq\frac{\mathbb{E}[X]}{a}$, $\footnotesize{\mathbb{E}[X] ~ \text{Valore atteso di una variabile casuale } X\geq 0, \forall a}$ \
Per $a = 2\mathbb{E}[X]$ la disuguaglianza diventa $P(X\geq 2\mathbb{E}[X])\leq\frac{\mathbb{E}[X]}{2\mathbb{E}[X]} = P(X\geq 2\mathbb{E}[X])\leq\frac{1}{2}$.

Questa disuguaglianza e' alla base del secondo algoritmo infatti, se $X$ e' la variabile casuale che all'i-esima iterazione di $\text{LVQuicksort(S)}$ ordina la sequenza $S$ in $x_i$ confronti e $k > 0 \in \real$ avremo:

In [ ]:
def mc_quicksort(s, k):
    global count, limit
    
    n = len(s)
    u = n * math.log2(n) # estimated expected value of the number of comparisons necessary to order s, E[X] = nlog(n)
    limit = 2*u
    
    for i in range(k):
        s_copy = list(s)
        count = 0
        
        if not r_quick_sort(s_copy, 0, n-1): # if x > 2(nlog(n)) -> discard iteration
            continue
        else:
            return s_copy

Come annunciato poco fa, essendo un algoritmo di tipologia **Monte Carlo**, $\text{MCQuicksort(S, k)}$ potrebbe fallire e non ordinare la sequenza $S$, infatti nel caso in cui tutte le $k$ iterazioni ordinassero la sequenza $S$ in un numero di confronti superiore a $2\hat\mu$, allora la procedura di ordinamento fallisce. Nel caso peggiore $\text{MCQuicksort(S, k)}$ ordina la sequenza nell'**ultima iterazione** con un costo che non e' superiore a $2k\hat\mu\approx2k\mathbb{E}[X]$ e quindi sempre in $O(nlogn)$ confronti. Tutto sta nello scegliere un $k$ opportuno per rendere trascurabile la probabilita' di _insuccesso_ dell'algoritmo.